In [9]:
!pip install monai wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 37.6 MB/s eta 0:00:0000:01


In [1]:
import os

# Check dataset is mounted correctly
tensor_path = '/kaggle/input/datasets/arinehkhachikian/adni-alzheimer-tensors/'
files = os.listdir(tensor_path)
print(f"Files/folders found: {len(files)}")
print(f"First 15: {files[:15]}")

Files/folders found: 17
First 15: ['best_model_binary.pt', 'resnet_10_23dataset.pth', 'best_model_final.pt', 'val_binary_multi.csv', 'best_model_no_scores.pt', 'val.csv', 'val_binary.csv', 'test_binary.csv', 'tensors', 'train_binary_multi.csv', 'best_model_multimodal.pt', 'master_labels.csv', 'train.csv', 'test.csv', 'test_binary_multi.csv']


In [2]:
import pandas as pd

train_df = pd.read_csv(f'{tensor_path}/train_binary.csv')
val_df = pd.read_csv(f'{tensor_path}/val_binary.csv')
test_df = pd.read_csv(f'{tensor_path}/test_binary.csv')

print(f"Train: {len(train_df)} subjects.")
print(f"Validation: {len(val_df)} subjects.")
print(f"Test: {len(test_df)} subjects.")

Train: 230 subjects.
Validation: 50 subjects.
Test: 50 subjects.


In [3]:
from torch.utils.data import Dataset, DataLoader

class TensorDataset(Dataset):
    def __init__(self, dataframe, tensors_dir):
        self.data = dataframe.reset_index(drop=True)
        self.tensors_dir = tensors_dir

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        ptid = row['PTID']
        label = int(row['label'])
        tensor_path = os.path.join(self.tensors_dir, f'{ptid}.pt')
        sample = torch.load(tensor_path, weights_only=False)
        volume = sample['volume']
        return volume, label

In [5]:
# resnet.py

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
import math
from functools import partial

__all__ = [
    'ResNet', 'resnet10', 'resnet18', 'resnet34', 'resnet50', 'resnet101',
    'resnet152', 'resnet200'
]


def conv3x3x3(in_planes, out_planes, stride=1, dilation=1):
    # 3x3x3 convolution with padding
    return nn.Conv3d(
        in_planes,
        out_planes,
        kernel_size=3,
        dilation=dilation,
        stride=stride,
        padding=dilation,
        bias=False)


def downsample_basic_block(x, planes, stride, no_cuda=False):
    out = F.avg_pool3d(x, kernel_size=1, stride=stride)
    zero_pads = torch.Tensor(
        out.size(0), planes - out.size(1), out.size(2), out.size(3),
        out.size(4)).zero_()
    if not no_cuda:
        if isinstance(out.data, torch.cuda.FloatTensor):
            zero_pads = zero_pads.cuda()

    out = Variable(torch.cat([out.data, zero_pads], dim=1))

    return out


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, dilation=1, downsample=None):
        super(BasicBlock, self).__init__()
        self.conv1 = conv3x3x3(inplanes, planes, stride=stride, dilation=dilation)
        self.bn1 = nn.BatchNorm3d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3x3(planes, planes, dilation=dilation)
        self.bn2 = nn.BatchNorm3d(planes)
        self.downsample = downsample
        self.stride = stride
        self.dilation = dilation

    def forward(self, x):
        residual = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            residual = self.downsample(x)

        out += residual
        out = self.relu(out)

        return out


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, inplanes, planes, stride=1, dilation=1, downsample=None):
        super(Bottleneck, self).__init__()
        self.conv1 = nn.Conv3d(inplanes, planes, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm3d(planes)
        self.conv2 = nn.Conv3d(
            planes, planes, kernel_size=3, stride=stride, dilation=dilation, padding=dilation, bias=False)
        self.bn2 = nn.BatchNorm3d(planes)
        self.conv3 = nn.Conv3d(planes, planes * 4, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm3d(planes * 4)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride
        self.dilation = dilation

    def forward(self, x):
        residual = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            residual = self.downsample(x)

        out += residual
        out = self.relu(out)

        return out


class ResNet(nn.Module):

    def __init__(self,
                 block,
                 layers,
                 sample_input_D,
                 sample_input_H,
                 sample_input_W,
                 num_seg_classes,
                 shortcut_type='B',
                 no_cuda = False):
        self.inplanes = 64
        self.no_cuda = no_cuda
        super(ResNet, self).__init__()
        self.conv1 = nn.Conv3d(
            1,
            64,
            kernel_size=7,
            stride=(2, 2, 2),
            padding=(3, 3, 3),
            bias=False)
            
        self.bn1 = nn.BatchNorm3d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool3d(kernel_size=(3, 3, 3), stride=2, padding=1)
        self.layer1 = self._make_layer(block, 64, layers[0], shortcut_type)
        self.layer2 = self._make_layer(
            block, 128, layers[1], shortcut_type, stride=2)
        self.layer3 = self._make_layer(
            block, 256, layers[2], shortcut_type, stride=1, dilation=2)
        self.layer4 = self._make_layer(
            block, 512, layers[3], shortcut_type, stride=1, dilation=4)

        self.conv_seg = nn.Sequential(
                                        nn.ConvTranspose3d(
                                        512 * block.expansion,
                                        32,
                                        2,
                                        stride=2
                                        ),
                                        nn.BatchNorm3d(32),
                                        nn.ReLU(inplace=True),
                                        nn.Conv3d(
                                        32,
                                        32,
                                        kernel_size=3,
                                        stride=(1, 1, 1),
                                        padding=(1, 1, 1),
                                        bias=False), 
                                        nn.BatchNorm3d(32),
                                        nn.ReLU(inplace=True),
                                        nn.Conv3d(
                                        32,
                                        num_seg_classes,
                                        kernel_size=1,
                                        stride=(1, 1, 1),
                                        bias=False) 
                                        )

        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                m.weight = nn.init.kaiming_normal(m.weight, mode='fan_out')
            elif isinstance(m, nn.BatchNorm3d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()

    def _make_layer(self, block, planes, blocks, shortcut_type, stride=1, dilation=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            if shortcut_type == 'A':
                downsample = partial(
                    downsample_basic_block,
                    planes=planes * block.expansion,
                    stride=stride,
                    no_cuda=self.no_cuda)
            else:
                downsample = nn.Sequential(
                    nn.Conv3d(
                        self.inplanes,
                        planes * block.expansion,
                        kernel_size=1,
                        stride=stride,
                        bias=False), nn.BatchNorm3d(planes * block.expansion))

        layers = []
        layers.append(block(self.inplanes, planes, stride=stride, dilation=dilation, downsample=downsample))
        self.inplanes = planes * block.expansion
        for i in range(1, blocks):
            layers.append(block(self.inplanes, planes, dilation=dilation))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.conv_seg(x)

        return x

def resnet10(**kwargs):
    """Constructs a ResNet-18 model.
    """
    model = ResNet(BasicBlock, [1, 1, 1, 1], **kwargs)
    return model


def resnet18(**kwargs):
    """Constructs a ResNet-18 model.
    """
    model = ResNet(BasicBlock, [2, 2, 2, 2], **kwargs)
    return model


def resnet34(**kwargs):
    """Constructs a ResNet-34 model.
    """
    model = ResNet(BasicBlock, [3, 4, 6, 3], **kwargs)
    return model


def resnet50(**kwargs):
    """Constructs a ResNet-50 model.
    """
    model = ResNet(Bottleneck, [3, 4, 6, 3], **kwargs)
    return model


def resnet101(**kwargs):
    """Constructs a ResNet-101 model.
    """
    model = ResNet(Bottleneck, [3, 4, 23, 3], **kwargs)
    return model


def resnet152(**kwargs):
    """Constructs a ResNet-101 model.
    """
    model = ResNet(Bottleneck, [3, 8, 36, 3], **kwargs)
    return model


def resnet200(**kwargs):
    """Constructs a ResNet-101 model.
    """
    model = ResNet(Bottleneck, [3, 24, 36, 3], **kwargs)
    return model

In [6]:
class TensorDataset(Dataset):
    def __init__(self, dataframe, tensors_dir):
        self.data = dataframe.reset_index(drop=True)
        self.tensors_dir = tensors_dir

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        ptid = row['PTID']
        label = int(row['label'])
        tensor_path = os.path.join(self.tensors_dir, f'{ptid}.pt')
        sample = torch.load(tensor_path, weights_only=False)
        volume = sample['volume']
        return volume, label

In [7]:
tensors_dir = '/kaggle/input/datasets/arinehkhachikian/adni-alzheimer-tensors/tensors/tensors'

train_dataset = TensorDataset(train_df, tensors_dir)
val_dataset = TensorDataset(val_df, tensors_dir)

train_dataloader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=2, shuffle=False)

In [10]:
# model.py

import torch
import torch.nn as nn
from monai.networks.nets import DenseNet121, ViT
import sys
import os

def get_densenet():
    model = DenseNet121(
        spatial_dims = 3,               # volumetric (1, 96, 112, 96)
        in_channels = 1,                # grayscale MRI (1, 96, 112, 96)
        out_channels = 2
    )
    return model

def get_vit():
    model = ViT(
        in_channels = 1,                 
        img_size = (128, 128, 128),     # dim. of input image
        patch_size = (16, 16, 16),      # dim. of patch size
        hidden_size = 768,              # dim. of hidden layer
        mlp_dim = 3072,                 # dim. of feedforward layer
        num_layers = 12,                # # of transformer blocks
        num_heads = 12,                 # # of attention heads
        num_classes = 3,                # # of classes if classification is used
        classification = True           # Bool, determines if classification is used
    )
    return model

def get_medicalnet(pretrained_path, num_classes=3, device='cpu'):
    # Load ResNet10 architecture
    model = resnet10(
        sample_input_W = 128,
        sample_input_H = 128,
        sample_input_D = 128,
        shortcut_type = 'B',
        no_cuda=(device == 'cpu'),
        num_seg_classes = num_classes
    )

    # Load pretrained weights
    checkpoint = torch.load(pretrained_path, map_location = device)
    state_dict = checkpoint['state_dict']

    # Remove 'module.' prefix if present (from DataParallel training)
    new_state_dict = {}
    for k, v in state_dict.items():
        name = k.replace('module.', '')
        new_state_dict[name] = v

    # Load weights with strict=False to allow mismatched final layer
    model.load_state_dict(new_state_dict, strict=False)

    # Replace final segmentation layer with classification layer
    model.conv_seg = nn.Sequential(
        nn.AdaptiveAvgPool3d((1, 1, 1)),
        nn.Flatten(),
        nn.Linear(512, num_classes)
    )

    return model

def get_medicalnet_multimodal(pretrained_path, num_clinical_features=3, num_classes=2, device='cpu'):
    # Load base ResNet10
    backbone = resnet10(
        sample_input_W=128,
        sample_input_H=128,
        sample_input_D=128,
        shortcut_type='B',
        no_cuda=(device == 'cpu'),
        num_seg_classes=num_classes
    )

    # Load pretrained weights
    checkpoint = torch.load(pretrained_path, map_location=device, weights_only=False)
    state_dict = checkpoint['state_dict']
    new_state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
    backbone.load_state_dict(new_state_dict, strict=False)

    # Build multimodal model
    class MultimodalResNet(nn.Module):
        def __init__(self, backbone, num_clinical_features, num_classes):
            super().__init__()
            # Image feature extractor — everything except final layer
            self.image_encoder = nn.Sequential(
                backbone.conv1,
                backbone.bn1,
                backbone.relu,
                backbone.maxpool,
                backbone.layer1,
                backbone.layer2,
                backbone.layer3,
                backbone.layer4,
                nn.AdaptiveAvgPool3d((1, 1, 1)),
                nn.Flatten()
            )
            # Clinical feature encoder
            self.clinical_encoder = nn.Sequential(
                nn.Linear(num_clinical_features, 32),
                nn.ReLU(),
                nn.Linear(32, 32)
            )
            # Fusion classifier
            self.classifier = nn.Sequential(
                nn.Linear(512 + 32, 128),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(128, num_classes)
            )

        def forward(self, image, clinical):
            img_features = self.image_encoder(image)
            clin_features = self.clinical_encoder(clinical)
            combined = torch.cat([img_features, clin_features], dim=1)
            return self.classifier(combined)

    model = MultimodalResNet(backbone, num_clinical_features, num_classes)
    return model

In [11]:
# train.py

import torch
import torch.nn as nn
import wandb
import os

def train_one_epoch(model, train_loader, optimizer, criterion, device):
    model.train()
 
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

    avg_loss = running_loss / len(train_loader)
    accuracy = correct / total
    return avg_loss, accuracy


def validate_one_epoch(model, val_loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
                
            outputs = model(images)
            loss = criterion(outputs, labels)
                        
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

    avg_loss = running_loss / len(val_loader)
    accuracy = correct / total
    return avg_loss, accuracy


def train(model, train_loader, val_loader, optimizer, criterion, scheduler, device, num_epochs, patience):
    wandb.init(project="alzheimer-mri-progression", config={
        "epochs": num_epochs,
        "learning_rate": 1e-5,
        "batch_size": 2,
        "architecture": "MedicalNet-Binary"
    })

    best_val_acc = 0
    epochs_no_improve = 0

    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)

        wandb.log({
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
            "epoch": epoch
        })

        print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_model_binary.pt')
            print(f"New best model saved with val accuracy: {val_acc:.4f}")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            break

        scheduler.step()

In [ ]:
# Set up and run training

from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

pretrained_path = '/kaggle/input/datasets/arinehkhachikian/adni-alzheimer-tensors/resnet_10_23dataset.pth'
model = get_medicalnet(pretrained_path, num_classes=2, device='cuda').to(device)

class_counts = torch.tensor([136.0, 94.0])  # CN, Dementia
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum()
class_weights = class_weights.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = AdamW(model.parameters(), lr=1e-5, weight_decay=1e-5)
scheduler = CosineAnnealingLR(optimizer, T_max=100, eta_min=1e-6)

train(model, train_dataloader, val_dataloader, optimizer, criterion, scheduler, device, num_epochs=100, patience=20)

In [23]:
from IPython.display import FileLink
FileLink('best_model_binary.pt')

/kaggle/working/best_model_no_scores.pt